In [1]:
!pip install groq

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Anees_Curated_Word_Bank.csv to Anees_Curated_Word_Bank (1).csv


In [3]:
import pandas as pd

df = pd.read_csv("Anees_Curated_Word_Bank.csv")

df.head()

,الكلمة,المعنى المرجعي,مصدر المعنى
0,السَّنا,نور,معجم الرياض
1,الشَّفَق,حمرة تظهر في الأفق من بقية ضوء الشمس بعد غروبها‏,معجم الرياض
2,الغُروب,أُفُول‏,معجم الرياض
3,الفَجر,أول وقت النهار وهو وقت ظهور ضوء خفيف,معجم الرياض
4,النَّدى,بخار المَاء يتكاثف فِي طَبَقَات الجو الْبَارِد...,معجم الرياض


In [4]:
print("Number of words:", len(df))
print("Columns:", df.columns.tolist())

Number of words: 100
Columns: ['الكلمة', 'المعنى المرجعي', 'مصدر المعنى']


In [10]:
import os
key = os.getenv("GROQ_API_KEY_NEW")

In [11]:
print(key)

None


In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY_NEW")

client = Groq(api_key=api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [12]:
class AneesWordGame:

    def __init__(self, client, age_range="6 إلى 11 سنة", debug=False):
        self.client = client
        self.age_range = age_range
        self.debug = debug

    def ask_groq(self, prompt, temperature=0.3, reasoning_effort="low"):
        response = self.client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            reasoning_effort=reasoning_effort
        )
        return response.choices[0].message.content.strip()

    def _fix_gender_agreement(self, question):
        fixes = {
            "ما هي الشعور": "ما هو الشعور",
            "ما هو الظاهرة": "ما هي الظاهرة",
            "ما هي الشيء": "ما هو الشيء",
            "ما هو الحركة": "ما هي الحركة",
            "ما هو الصفة": "ما هي الصفة",
            "ما هو الكلمة": "ما هي الكلمة",
            "ما هي المكان": "ما هو المكان",
            "ما هي الصوت": "ما هو الصوت",
        }

        for wrong, right in fixes.items():
            question = question.replace(wrong, right)

        return question

    def generate_question(self, word, meaning, max_attempts=3):

        def normalize(text):
            text = text.replace('ال', '', 1) if text.startswith('ال') else text
            diacritics = 'ًٌٍَُِّْ'
            return ''.join(
                c for c in text
                if c not in diacritics
            ).strip()

        clean_word = normalize(word)

        for attempt in range(max_attempts):

            prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
حوّل المعنى المرجعي إلى سؤال قصير وواضح يستطيع الطفل معرفة الكلمة
الصحيحة من خلاله.

لا تجعل صيغة السؤال ثابتة لكل الكلمات؛ اختر صياغة تناسب نوع الكلمة
ووظيفتها ومعناها.

قواعد مهمة:

- يجب أن تكون بداية السؤال صحيحة نحويًا.
- استخدم "ما هي" إذا كان الاسم الذي يليها مؤنثًا.
- استخدم "ما هو" إذا كان الاسم الذي يليها مذكرًا.
- لا تستخدم "ما هو" قبل اسم مؤنث، ولا "ما هي" قبل اسم مذكر.
- يجب أن تكون الجملة صحيحة نحويًا بالكامل.
- استخدم عبارة أو فئة عامة تصف نوع الكلمة، وتكون مختلفة عن الكلمة الصحيحة.
- صف مشهدًا أو موقفًا ملموسًا يستطيع الطفل تخيله بسهولة.
- لا تذكر الكلمة الصحيحة أو أي جزء منها في السؤال إطلاقًا.
- لا تستخدم الكلمة الصحيحة داخل مثال أو وصف أو موقف.
- استخدم المعنى المرجعي كأساس للسؤال.
- يمكنك إضافة وصف أو موقف يساعد الطفل على فهم الكلمة، حتى لو لم يكن
  مذكورًا حرفيًا في المعنى المرجعي، بشرط ألا يغيّر المعنى الأساسي
  للكلمة أو يجعل إجابة أخرى صحيحة.
- يجب أن يحافظ السؤال على نوع الكلمة النحوي ومعناها كما وردا في الكلمة
  والمعنى المرجعي.
- لا تغيّر نوع الكلمة عند صياغة السؤال.
- إذا كانت الكلمة اسمًا، يجب أن يكون المطلوب في السؤال اسمًا وليس فعلًا.
- إذا كانت الكلمة فعلًا، يجب أن يكون المطلوب في السؤال فعلًا.
- إذا كانت الكلمة صفة، يجب أن يكون المطلوب في السؤال صفة.
- لا تستنتج أن الكلمة فعل لمجرد أن المعنى المرجعي يصف فعلًا.
- لا تستنتج أن الكلمة اسم لمجرد أن المعنى المرجعي يصف شيئًا.
- يجب أن تكون الإجابة المتوقعة هي الكلمة نفسها من حيث نوعها وصيغتها.
- لا توسّع المعنى المرجعي إلى معنى آخر غير موجود فيه.
- لا تضف معاني مجازية أو استخدامات أخرى للكلمة لم يذكرها المعنى المرجعي.
- إذا كانت الكلمة اسمًا لشيء، اجعل السؤال عن الشيء نفسه، وليس عن
  استخدامه أو أثره أو فعل مرتبط به.
- إذا كانت الكلمة تدل على مكان، اجعل السؤال عن المكان نفسه.
- إذا كانت الكلمة تدل على شعور، اجعل السؤال عن الشعور نفسه.
- إذا كانت الكلمة تدل على صوت، اجعل السؤال عن الصوت نفسه.
- إذا كانت الكلمة تدل على فعل، اجعل السؤال عن الفعل نفسه.
- إذا كانت الكلمة تدل على صفة أو ملمس، اجعل السؤال عن الصفة أو الملمس نفسه.
- لا تحوّل الكلمة إلى شيء آخر مرتبط بها.
  مثال: إذا كانت الكلمة تدل على مصدر للماء، فلا تسأل عنها باعتبارها الماء نفسه.
- يجب أن يتضمن السؤال تفاصيل مميزة مرتبطة بالمعنى المرجعي للكلمة،
  بحيث لا تكون الكلمات العامة أو القريبة منها إجابات صحيحة.
- تجنب الأسئلة العامة جدًا التي يمكن أن تنطبق على أكثر من شيء.
- إذا كانت الكلمة تصف مكانًا أو شيئًا له خصائص مشتركة مع كلمات أخرى،
  استخدم في السؤال الخاصية الأكثر تمييزًا للكلمة كما وردت في المعنى المرجعي.
- اجعل الوصف في السؤال كافيًا لتمييز الكلمة الصحيحة عن الكلمات المرتبطة
  بها في المجال نفسه.
- لا تستخدم أوصافًا عامة جدًا إذا كانت تجعل أكثر من خيار ممكنًا كإجابة.
- عندما يكون المعنى المرجعي يحتوي على خاصية مميزة، اجعل هذه الخاصية
  واضحة في السؤال بدل الاكتفاء بوصف عام.

قبل إخراج السؤال، تحقق من ثلاثة أشياء:
1. هل السؤال يطلب نفس نوع الكلمة؟
2. هل السؤال يعتمد على المعنى المرجعي فقط؟
3. هل توجد كلمة أخرى يمكن أن تكون إجابة صحيحة بدل الكلمة المطلوبة؟

إذا كانت الإجابة عن أي منها "لا"، أعد صياغة السؤال قبل إخراجه.

- قبل إخراج السؤال، راجع صيغته نحويًا وتأكد من صحة التذكير والتأنيث
  والمفرد والجمع.

أمثلة على الصياغة الصحيحة حسب نوع الكلمة:

- إذا كانت الكلمة فعلًا:
  "ما هي الحركة التي يقوم بها الطفل عندما يرتفع عن الأرض؟"

- إذا كانت الكلمة اسم مكان:
  "ما هو المكان الذي تخرج منه المياه من باطن الأرض؟"

- إذا كانت الكلمة صفة:
  "ما هي الصفة التي تصف الشيء اللين عند لمسه؟"

- إذا كانت الكلمة صوتًا:
  "ما هو الصوت الذي يصدر عند تحرك أوراق الشجر؟"

- إذا كانت الكلمة اسم شيء:
  "ما هو المصباح الذي يُعلّق أو يُحمل لإضاءة المكان؟"

مهم:
هذه الأمثلة تشرح طريقة صياغة السؤال فقط، وليست كلمات يجب استخدامها.
عند إنشاء السؤال، استخدم الكلمة والمعنى المرجعي الموجودين في الأعلى
ولا تعتمد على الأمثلة كمصدر للمعنى.

- استخدم لغة عربية فصحى بسيطة مناسبة للأطفال من عمر 6 إلى 11 سنة.
- اجعل السؤال واضحًا وله إجابة صحيحة واحدة فقط.
- لا تكتب خيارات.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

            question = self.ask_groq(
                prompt,
                temperature=0.3 + attempt * 0.1
            )

            question = question.strip()
            question = self._fix_gender_agreement(question)

            clean_question = normalize(question)

            if clean_word not in clean_question:
                return question

            if self.debug:
                print(
                    f"[تحذير] السؤال سرّب الكلمة، "
                    f"إعادة المحاولة {attempt + 1}: {question}"
                )

        fallback_prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
اكتب سؤالًا قصيرًا يصف الكلمة بصفاتها أو وظيفتها أو مكانها مباشرة،
بحيث يستطيع الطفل معرفة الكلمة من الوصف.

الشروط:
- لا تذكر الكلمة الصحيحة أو أي جزء منها إطلاقًا.
- لا تستخدم الكلمة الصحيحة داخل وصف أو مثال.
- لا تستخدم "ما معنى كلمة".
- لا تجعل السؤال عن المعنى المرجعي نفسه.
- اجعل الوصف يؤدي إلى الكلمة الصحيحة كإجابة.
- حافظ على نوع الكلمة ووظيفتها.
- لا تغيّر نوع الكلمة عند صياغة السؤال.
- إذا كانت الكلمة اسمًا، يجب أن يكون المطلوب في السؤال اسمًا وليس فعلًا.
- إذا كانت الكلمة فعلًا، يجب أن يكون المطلوب في السؤال فعلًا.
- إذا كانت الكلمة صفة، يجب أن يكون المطلوب في السؤال صفة.
- لا تضف معنى آخر للكلمة غير موجود في المعنى المرجعي.
- يجب أن يتضمن السؤال تفاصيل مميزة مرتبطة بالمعنى المرجعي.
- تجنب الأسئلة العامة التي يمكن أن تنطبق على أكثر من كلمة.
- استخدم الخاصية الأكثر تمييزًا للكلمة عندما يكون ذلك ممكنًا.
- اجعل السؤال يؤدي إلى إجابة واحدة واضحة فقط.
- ابدأ بـ"ما هو" أو "ما هي" بما يتوافق مع الاسم الذي يلي أداة السؤال.
- تأكد أن الجملة صحيحة نحويًا.
- استخدم لغة عربية فصحى بسيطة مناسبة لعمر 6 إلى 11 سنة.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        question = self.ask_groq(
            fallback_prompt,
            temperature=0.4
        )

        question = question.strip()
        question = self._fix_gender_agreement(question)

        clean_question = normalize(question)

        if clean_word not in clean_question:

            if self.debug:
                print(f"[أسلوب بديل] نجح: {question}")

            return question

        if self.debug:
            print(f"[أسلوب بديل] لسا مسرب: {question}")

        final_prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
اكتب سؤالًا قصيرًا يصف معنى الكلمة الصحيحة بطريقة تجعل الكلمة الصحيحة
هي الإجابة الوحيدة الممكنة.

مهم جدًا:
- يجب أن تكون إجابة السؤال هي الكلمة الصحيحة "{word}".
- لا تذكر الكلمة الصحيحة "{word}" في السؤال إطلاقًا.
- لا تذكر أي جزء منها.
- لا تذكر الكلمة الصحيحة داخل مثال أو وصف.
- لا تستخدم صيغة "ما معنى كلمة".
- لا تجعل المعنى المرجعي نفسه هو الإجابة.
- حافظ على نوع الكلمة ووظيفتها.
- لا تغيّر نوع الكلمة عند صياغة السؤال.
- إذا كانت الكلمة اسمًا، يجب أن يكون المطلوب في السؤال اسمًا وليس فعلًا.
- إذا كانت الكلمة فعلًا، يجب أن يكون المطلوب في السؤال فعلًا.
- إذا كانت الكلمة صفة، يجب أن يكون المطلوب في السؤال صفة.
- لا تضف معنى آخر للكلمة غير موجود في المعنى المرجعي.
- استخدم الخاصية الأكثر تمييزًا للكلمة في المعنى المرجعي.
- تجنب الوصف العام الذي يمكن أن ينطبق على كلمات أخرى.
- اجعل السؤال يؤدي إلى إجابة واحدة واضحة فقط.
- ابدأ بـ"ما هو" أو "ما هي" بما يتوافق مع الاسم الذي يلي أداة السؤال.
- تأكد من صحة التذكير والتأنيث.
- تأكد أن الجملة صحيحة نحويًا بالكامل.
- استخدم لغة عربية فصحى بسيطة مناسبة لطفل من عمر 6 إلى 11 سنة.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        question = self.ask_groq(
            final_prompt,
            temperature=0.3
        )

        question = question.strip()
        question = self._fix_gender_agreement(question)

        clean_question = normalize(question)

        if self.debug:
            status = (
                "نجح"
                if clean_word not in clean_question
                else "لسا مسرب"
            )

            print(
                f"[المحاولة الأخيرة] {status}: {question}"
            )

        return question

    def generate_distractors(self, word, meaning, question):

        prompt = f"""
أنت مساعد تعليمي في لعبة الكلمات في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

المهمة:
أنشئ 3 كلمات مشتتة مناسبة للسؤال.

الهدف:
اجعل الخيارات تبدو معقولة للطفل، لكن يجب أن تكون هناك إجابة صحيحة واحدة فقط.
يجب ألا تكون الكلمات المشتتة قريبة جدًا من معنى الكلمة الصحيحة لدرجة تجعل الطفل
يتردد بين أكثر من خيار.

القواعد الأساسية:

1. كل كلمة مشتتة يجب أن تكون مختلفة بوضوح في المعنى عن الكلمة الصحيحة.

2. لا تستخدم:
- مرادفًا للكلمة الصحيحة.
- شبه مرادف قريبًا جدًا من الكلمة الصحيحة.
- كلمة يمكن أن تحل محل الكلمة الصحيحة في السؤال وتحافظ على المعنى.
- كلمة من نفس العائلة الدلالية إذا كانت ستجعلها تبدو إجابة صحيحة.
- كلمة من نفس الجذر أو صيغة مشتقة من الكلمة الصحيحة.
- فعلًا أو اسمًا أو صفة مرتبطة بالكلمة الصحيحة ارتباطًا يجعلها إجابة محتملة.

مثال:

الكلمة الصحيحة:
تعاون

لا تستخدم:
مساعدة
مشاركة
شارك
تفاعل

لأنها قريبة جدًا من معنى التعاون وقد تجعل الطفل يحتار.

3. في المقابل، يمكن أن تكون الكلمة المشتتة مرتبطة بالسياق العام للسؤال،
لكن يجب أن يكون معناها مختلفًا بوضوح عن الكلمة الصحيحة.

مثال:

الكلمة الصحيحة:
تعاون

خيارات مناسبة:
مسابقة
رحلة
مغامرة

هذه الكلمات مناسبة لعالم الأطفال والأنشطة، لكنها لا تعني التعاون.

4. لا تجعل المشتتات بعيدة جدًا أو عشوائية.

لا تستخدم كلمات مثل:
تفاحة
سيارة
قمر

إذا لم يكن لها أي علاقة بالسياق.

5. لا تحاول جعل الكلمات الثلاث متشابهة في المعنى مع بعضها.
الأولوية هي أن تكون كل كلمة مختلفة بوضوح عن الكلمة الصحيحة.

6. يجب أن تكون الكلمات المشتتة من نفس النوع النحوي للكلمة الصحيحة قدر الإمكان:
- إذا كانت الكلمة الصحيحة اسمًا، استخدم أسماء.
- إذا كانت الكلمة الصحيحة فعلًا، استخدم أفعالًا.
- إذا كانت الكلمة الصحيحة صفة، استخدم صفات.
- إذا كانت الكلمة الصحيحة صوتًا، استخدم أسماء أصوات.
ولا تغيّر نوع الكلمة فقط لجعلها مشتتًا.

7. يجب أن تكون كل كلمة مناسبة لطفل من عمر 6 إلى 11 سنة.

8. يجب أن تكون كل كلمة عربية صحيحة ومستخدمة بشكل طبيعي.

9. لا تستخدم:
- كلمات مصطنعة.
- كلمات نادرة جدًا أو غير مناسبة لعمر الطفل.
- كلمات ناقصة.
- كلمات مركبة بشكل غير طبيعي.
- صيغًا مشتقة بشكل غير صحيح.

10. كل مشتت يجب أن يكون كلمة مستقلة واحدة فقط.

قبل إخراج الكلمات، راجع كل خيار واسأل نفسك:

- هل يمكن للطفل أن يختاره باعتباره الإجابة الصحيحة؟
- هل يمكن أن يحل محل الكلمة الصحيحة في السؤال؟
- هل هو مرادف أو شبه مرادف واضح؟
- هل هو قريب جدًا من المعنى الأساسي للكلمة الصحيحة؟

إذا كانت الإجابة "نعم" على أي منها، استبدل الكلمة.

مهم:
نحن لا نريد مشتتات بعيدة جدًا عن الموضوع،
ولا نريد مشتتات شبه مرادفة.
نريد كلمات معقولة في سياق السؤال ولكن مختلفة بوضوح في المعنى.

أمثلة:

الكلمة الصحيحة:
ثقة

مشتتات مناسبة:
شجاعة
فخر
تفاؤل

الكلمة الصحيحة:
تعاون

مشتتات مناسبة:
مسابقة
رحلة
مغامرة

الكلمة الصحيحة:
مخيم

مشتتات مناسبة:
فندق
منتزه
متحف

الكلمة الصحيحة:
بريق

مشتتات مناسبة:
ظل
ضباب
صوت

مهم:
هذه الأمثلة لتوضيح مستوى التشابه المطلوب فقط.
لا تستخدمها تلقائيًا، واعتمد على الكلمة والمعنى والسؤال الموجودين في الأعلى.

- أرجع 3 كلمات فقط.
- كل كلمة في سطر مستقل.
- لا تكتب أرقامًا.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        response = self.ask_groq(
            prompt,
            temperature=0.4
        )

        distractors = [
            line.strip()
            for line in response.splitlines()
            if line.strip()
        ]

        return distractors[:3]

    def verify_options(self, word, meaning, question, options):

        options_text = "\n".join(
            f"{i}: {opt}"
            for i, opt in enumerate(options)
        )

        prompt = f"""
أنت نظام تحقق للعبة الكلمات في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

الخيارات:
{options_text}

مهمتك:
تحقق من الخيارات الأربعة.

القواعد:

1. الخيار 0 هو الكلمة الصحيحة ويجب أن يكون PASS دائمًا.

2. الخيار الآخر يكون REJECT فقط في الحالات التالية:

- إذا كان يمكن اعتباره إجابة صحيحة فعلًا لنفس السؤال.
- إذا كان مرادفًا واضحًا ومباشرًا للكلمة الصحيحة.
- إذا كان يحمل نفس المعنى الأساسي للكلمة الصحيحة بدرجة كبيرة جدًا
  بحيث لا يستطيع الطفل التمييز بينهما.

3. لا ترفض الخيار لمجرد أنه:
- من نفس المجال.
- مرتبط بالموقف.
- مرتبط بالموضوع.
- قريب بشكل عام من السؤال.
- يمثل شعورًا مشابهًا لكنه ليس نفس الشعور.
- يمثل شيئًا موجودًا في نفس المكان أو الموقف.

أمثلة:

إذا كانت الكلمة الصحيحة "خجل":
"خوف" = PASS
"توتر" = PASS
"انزعاج" = PASS
لأنها مشاعر مختلفة وليست مرادفات مباشرة لـ"خجل".

إذا كانت الكلمة الصحيحة "برق":
"وميض" = PASS
"غيوم" = PASS
"رياح" = PASS
لأنها مرتبطة بالظاهرة لكنها ليست إجابة صحيحة مباشرة ولا مرادفات لـ"برق".

إذا كانت الكلمة الصحيحة "بداية":
"مستهل" = REJECT
"افتتاح" = REJECT
"انطلاقة" = REJECT
لأنها قريبة جدًا من معنى "بداية" ويمكن أن تكون إجابات بديلة معقولة.

إذا كانت الكلمة الصحيحة "جدول" بمعنى مجرى مائي صغير:
"دلو" = PASS
"قناة" = PASS
"حقل" = PASS
طالما أن السؤال يطلب مجرى الماء نفسه وليس أي شيء متعلق بالري.

4. لا تستخدم معيار "هل الخيار قريب من المعنى؟" وحده لرفض الخيار.

5. الهدف هو منع وجود إجابتين صحيحتين أو مرادفين واضحين،
وليس جعل جميع الخيارات بعيدة عن الكلمة الصحيحة.

6. كن متساهلًا في الحالات التي يوجد فيها مجرد ارتباط بالموضوع،
ومتشددًا فقط عندما يكون الخيار إجابة صحيحة أو مرادفًا واضحًا.

لا تكتب أي شرح أو تفكير.

أرجع فقط 4 أسطر بهذا الشكل بالضبط:

0: PASS
1: PASS أو REJECT
2: PASS أو REJECT
3: PASS أو REJECT
"""

        return self.ask_groq(
            prompt,
            temperature=0.0,
            reasoning_effort="low"
        )

    def get_rejected_indexes(self, verification_result, expected_count=4):

        import re

        rejected_indexes = []

        pattern = re.compile(
            r'^\s*(\d+)\s*:\s*(PASS|REJECT)\s*$'
        )

        matched_lines = 0

        for line in verification_result.splitlines():

            match = pattern.match(line.strip())

            if match:
                matched_lines += 1

                index = int(match.group(1))
                status = match.group(2)

                if status == "REJECT" and index != 0:
                    rejected_indexes.append(index)

        if matched_lines < expected_count:

            if self.debug:
                print(
                    f"[تحذير] التحقق رجّع "
                    f"{matched_lines}/{expected_count} أسطر — "
                    f"نرفض كل الخيارات احترازيًا"
                )

            rejected_indexes = list(range(1, expected_count))

        return rejected_indexes

    def generate_replacements(
        self,
        word,
        meaning,
        question,
        options,
        rejected_indexes
    ):

        options_text = "\n".join(
            f"{i}: {opt}"
            for i, opt in enumerate(options)
        )

        prompt = f"""
أنت مساعد تعليمي لمنصة أنيس للأطفال من عمر 6 إلى 11 سنة.

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

الخيارات الحالية:
{options_text}

أرقام الخيارات المرفوضة:
{rejected_indexes}

المهمة:
اقترح كلمات بديلة للخيارات المرفوضة.

الشروط:
- أنشئ كلمة بديلة مختلفة لكل خيار مرفوض.
- يجب أن تكون الكلمات البديلة معقولة في سياق السؤال،
  لكن مختلفة بوضوح في المعنى عن الكلمة الصحيحة.
- يجب ألا تكون إجابة صحيحة للسؤال.
- لا تستخدم مرادفات أو شبه مرادفات للكلمة الصحيحة.
- لا تستخدم كلمات يمكن أن تحل محل الكلمة الصحيحة في السؤال.
- لا تستخدم كلمات من نفس العائلة الدلالية إذا كانت ستسبب التباسًا.
- ليست مشتقة من الكلمة الصحيحة.
- مناسبة لطفل من عمر 6 إلى 11 سنة.
- لا تستخدم أي كلمة موجودة أصلًا في الخيارات.
- يجب أن تكون كل كلمة مختلفة عن الأخرى.

قواعد مهمة لجودة الكلمات:
- يجب أن تكون كل كلمة عربية صحيحة ومستخدمة بشكل طبيعي.
- لا تستخدم كلمات مصطنعة أو غير موجودة في الاستعمال العربي الطبيعي.
- لا تقطع الكلمة أو تحذف جزءًا منها.
- لا تغير بنية الكلمة بطريقة تجعلها ناقصة أو غير صحيحة.
- لا تدمج كلمتين معًا بدون مسافة أو تركيب عربي صحيح.
- لا تضف كلمات أو مقاطع عشوائية إلى كلمة موجودة.
- لا تستخدم صيغًا تبدو صحيحة شكليًا لكنها غير مستخدمة في العربية.
- استخدم كلمات مستقلة وواضحة.
- استخدم كلمات مناسبة ومفهومة لطفل من عمر 6 إلى 11 سنة.
- راجع كل كلمة قبل إخراجها وتأكد أنها كلمة عربية سليمة.

- أرجع عددًا من الكلمات يساوي عدد الخيارات المرفوضة.
- كل كلمة في سطر مستقل.
- لا تكتب أرقامًا.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        response = self.ask_groq(
            prompt,
            temperature=0.4
        )

        replacements = [
            line.strip()
            for line in response.splitlines()
            if line.strip()
        ]

        return replacements

    def generate_game(self, word, meaning, max_attempts=3):

        question = self.generate_question(
            word,
            meaning
        )

        distractors = self.generate_distractors(
            word,
            meaning,
            question
        )

        options = [word] + distractors[:3]

        verification_result = self.verify_options(
            word,
            meaning,
            question,
            options
        )

        rejected_indexes = self.get_rejected_indexes(
            verification_result
        )

        attempt = 1

        if self.debug:
            print(
                f"[محاولة {attempt}] "
                f"مرفوض: {rejected_indexes}"
            )
            print(verification_result)

        while rejected_indexes and attempt < max_attempts:

            attempt += 1

            replacements = self.generate_replacements(
                word,
                meaning,
                question,
                options,
                rejected_indexes
            )

            if len(replacements) < len(rejected_indexes) and self.debug:
                print(
                    f"[تحذير] طلبنا {len(rejected_indexes)} بدائل، "
                    f"رجع {len(replacements)}"
                )

            for index, replacement in zip(
                rejected_indexes,
                replacements
            ):
                options[index] = replacement

            verification_result = self.verify_options(
                word,
                meaning,
                question,
                options
            )

            rejected_indexes = self.get_rejected_indexes(
                verification_result
            )

            if self.debug:
                print(
                    f"[محاولة {attempt}] "
                    f"مرفوض: {rejected_indexes}"
                )
                print(verification_result)

        seen = set()
        duplicate_indexes = []

        for index, option in enumerate(options):

            normalized_option = option.strip().lower()

            if normalized_option in seen:
                duplicate_indexes.append(index)
            else:
                seen.add(normalized_option)

        if duplicate_indexes:

            if self.debug:
                print(
                    f"[تكرار] خيارات مكررة: "
                    f"{duplicate_indexes}"
                )

            replacements = self.generate_replacements(
                word,
                meaning,
                question,
                options,
                duplicate_indexes
            )

            if len(replacements) < len(duplicate_indexes) and self.debug:
                print(
                    f"[تحذير] طلبنا {len(duplicate_indexes)} بدائل للتكرار، "
                    f"رجع {len(replacements)}"
                )

            for index, replacement in zip(
                duplicate_indexes,
                replacements
            ):
                options[index] = replacement

            verification_result = self.verify_options(
                word,
                meaning,
                question,
                options
            )

            rejected_indexes = self.get_rejected_indexes(
                verification_result
            )

        import random

        correct_word = word
        shuffled_options = options[:]

        random.shuffle(shuffled_options)

        correct_index = shuffled_options.index(
            correct_word
        )

        return {
            "success": len(rejected_indexes) == 0,
            "word": word,
            "meaning": meaning,
            "question": question,
            "options": shuffled_options,
            "correct_index": correct_index,
            "verification": verification_result,
            "rejected_indexes": rejected_indexes,
            "attempts": attempt,
        }

In [13]:
game = AneesWordGame(client, debug=True)

In [14]:
distractors = game.generate_distractors(
    "نقيق",
    "صوت الضفدع",
    "ما هو الصوت الذي يصدره الضفدع؟"
)

print(distractors)

['زقزقة', 'عواء', 'صياح']


In [15]:
sample = df.sample(1)
for _, row in sample.iterrows():

    word = row["الكلمة"]
    meaning = row["المعنى المرجعي"]

    print("=" * 80)
    print(f"الكلمة: {word}")

    result = game.generate_game(word, meaning)

    print("السؤال:", result["question"])
    print("الخيارات:", result["options"])
    print("الإجابة:", result["word"])
    print("Success:", result["success"])
    print("Attempts:", result["attempts"])

الكلمة: تحد
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هو الفعل الذي يعني مواجهة صعبة ومحاولة التغلب عليها؟
الخيارات: ['سافر', 'قرأ', 'لعب', 'تحد']
الإجابة: تحد
Success: True
Attempts: 1


In [16]:
import pandas as pd

df = pd.read_csv("Anees_Curated_Word_Bank.csv")

print(f"Number of words: {len(df)}")
print(df.head())

Number of words: 100
     الكلمة                                     المعنى المرجعي  مصدر المعنى
0   السَّنا                                                نور  معجم الرياض
1  الشَّفَق   حمرة تظهر في الأفق من بقية ضوء الشمس بعد غروبها‏  معجم الرياض
2   الغُروب                                            أُفُول‏  معجم الرياض
3    الفَجر               أول وقت النهار وهو وقت ظهور ضوء خفيف  معجم الرياض
4   النَّدى  بخار المَاء يتكاثف فِي طَبَقَات الجو الْبَارِد...  معجم الرياض


In [17]:
sample = df.sample(10)

for _, row in sample.iterrows():

    word = row["الكلمة"]
    meaning = row["المعنى المرجعي"]

    print("=" * 80)
    print(f"الكلمة: {word}")

    result = game.generate_game(
        word,
        meaning,
        max_attempts=3
    )

    print("السؤال:", result["question"])
    print("الخيارات:", result["options"])
    print("الإجابة:", result["word"])
    print("Success:", result["success"])
    print("Attempts:", result["attempts"])

الكلمة: خجل
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هو الشعور الذي يَظهر عندما يتحول لون الوجه إلى الأحمر بسبب الإحراج؟
الخيارات: ['قوة', 'سعادة', 'فضول', 'خجل']
الإجابة: خجل
Success: True
Attempts: 1
الكلمة: مغامرة
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هي الكلمة التي تصف تجربة يخوضها الطفل بشجاعة ويتحمل فيها خطرًا كبيرًا ليستمتع بالاكتشاف؟
الخيارات: ['مسابقة', 'رحلة', 'مغامرة', 'لعبة']
الإجابة: مغامرة
Success: True
Attempts: 1
الكلمة: مَجرة
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هي التجمعات الضخمة التي تتكون من عدد هائل من النجوم والكواكب في الفضاء؟
الخيارات: ['مَجرة', 'صاروخ', 'رائد\u202fفضاء', 'محطة\u202fفضائية']
الإجابة: مَجرة
Success: True
Attempts: 1
الكلمة: سر 
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هو الشيء الذي يخفيه الطفل في قلبه ولا يريد أن يقوله للآخرين؟
الخيارات: ['فكرة', 'لعبة', 'حلم', 'سر ']
الإجابة: سر 
Success: True
Attempts: 1
الكلمة: مشاركة
[محاولة 1] مرفوض: []
0: PASS